# Art Basel 2025 Data Preprocessing

This notebook preprocesses the CSV data:
1. Fetches ticket prices from event links using Gemini
2. Standardizes address formats

Run each cell step by step and verify the results before proceeding.

## Setup

In [24]:
# Install required packages (run once)
!pip install google-generativeai pandas python-dotenv pyshorteners

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyshorteners: filename=pyshorteners-1.0.1-py3-none-any.whl size=17540 sha256=37fdb8b647f9d27a5334e34d52c1d700c9cd9baa0a58b68c16c0b1337deb38f7
  Stored in directory: /Users/jzamudio/Library/Caches/pip/wheels/30/e9/7c/06d5ea1c320274f44cc35321a75198e266b2d47b3092aa9b7e
Successfully built pyshorteners


In [46]:
import pandas as pd
import google.generativeai as genai
from dotenv import load_dotenv
import os
import time

# Load environment variables from .env.local (in parent directory)
# Try multiple paths to find the .env.local file
env_paths = [
    '../.env.local',      # If running from scripts folder
    '.env.local',         # If running from project root
    os.path.expanduser('~/.env.local')  # Home directory fallback
]

env_loaded = False
for env_path in env_paths:
    if os.path.exists(env_path):
        load_dotenv(env_path)
        print(f"✓ Loaded environment from: {env_path}")
        env_loaded = True
        break

if not env_loaded:
    print("⚠ No .env.local file found. You can set the API key manually below.")

# Configure Gemini
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

# If not found, you can set it manually here (uncomment and add your key):
# GEMINI_API_KEY = "your-api-key-here"

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found. Either:\n1. Create .env.local in the project root with GEMINI_API_KEY=your-key\n2. Uncomment the line above and paste your key directly")

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-2.0-flash-lite')

print("✓ Gemini API configured successfully")

✓ Loaded environment from: ../.env.local
✓ Gemini API configured successfully


## Load CSV Data

In [26]:
# Load the CSV
df = pd.read_csv('../art-basel-2025.csv')

print(f"Loaded {len(df)} events")
print(f"\nColumns: {list(df.columns)}")
df.head()

Loaded 172 events

Columns: ['ID', 'Event', 'Event Type', 'Start Date', 'End Date', 'Schedule', 'Neighborhood', 'Address', 'Notes', 'Ticket Price', 'Tickets Link']


,ID,Event,Event Type,Start Date,End Date,Schedule,Neighborhood,Address,Notes,Ticket Price,Tickets Link
0,1,"AI + Web3 $44,000 Pitch Competition & VC Netwo...",Networking,12/5/2025,12/5/2025,10AM - 12PM,NaN,NaN,Location Revealed to Ticket Holders,NaN,https://luma.com/pbs3n297
1,2,24th Iconic Sagamore Brunch,Party,12/6/2025,12/6/2025,9AM - 1PM,Miami Beach,Sagamore Hotel South Beach,NaN,NaN,https://www.eventbrite.com/e/the-24th-iconic-s...
2,3,33+ Hour Space Basel Closing Party,Party,12/7/2025,12/9/2025,NaN,Downtown,"34 NE 11th St, Miami, FL 33132",NaN,NaN,https://dice.fm/event/xenqkk-33-hour-space-bas...
3,4,AfriKin Art Fair,Art Show,12/1/2025,12/7/2025,11AM - 9PM,North Miami,"Maison AfriKin, Scott Galvin Community Center,...",NaN,NaN,https://afrikin.art/
4,5,Alo Wellness Sessions,Wellness,12/4/2025,12/7/2025,10AM - 11AM,Miami Beach,"SCOPE Art Show - 801 Ocean Dr, Miami Beach, FL...",Invite Only,NaN,https://scope-art.com/program


In [27]:
# Check current state of Ticket Price column
print("Ticket Price column stats:")
print(f"  - Empty/NaN: {df['Ticket Price'].isna().sum() + (df['Ticket Price'] == '').sum()}")
print(f"  - Has value: {df['Ticket Price'].notna().sum() - (df['Ticket Price'] == '').sum()}")
print(f"\nSample values:")
df[['Event', 'Ticket Price', 'Tickets Link']].head(10)

Ticket Price column stats:
  - Empty/NaN: 172
  - Has value: 0

Sample values:


,Event,Ticket Price,Tickets Link
0,"AI + Web3 $44,000 Pitch Competition & VC Netwo...",NaN,https://luma.com/pbs3n297
1,24th Iconic Sagamore Brunch,NaN,https://www.eventbrite.com/e/the-24th-iconic-s...
2,33+ Hour Space Basel Closing Party,NaN,https://dice.fm/event/xenqkk-33-hour-space-bas...
3,AfriKin Art Fair,NaN,https://afrikin.art/
4,Alo Wellness Sessions,NaN,https://scope-art.com/program
5,Aqua Art Miami,NaN,https://www.aquaartmiami.com/
6,Aqua Art Miami: VIP Opening,NaN,https://www.aquaartmiami.com
7,Arlo Wynwood: AFTER DARK,NaN,https://www.eventbrite.com/e/art-access-miami-...
8,Arlo Wynwood: Afternoon Art Exhibition,NaN,https://www.eventbrite.com/e/art-access-miami-...
9,Arlo Wynwood: COFFEE + BEATS,NaN,https://www.eventbrite.com/e/art-access-miami-...


## Helper Functions

In [32]:
import requests

def shorten_url_tinyurl(long_url):
    """
    Shortens a given URL using the TinyURL API.

    Args:
        long_url (str): The URL to be shortened.

    Returns:
        str: The shortened TinyURL, or an error message if the request fails.
    """
    api_url = f'http://tinyurl.com/api-create.php?url={long_url}'
    try:
        response = requests.get(api_url)
        response.raise_for_status()  # Raise an exception for bad status codes
        short_url = response.text
        return short_url
    except requests.exceptions.RequestException as e:
        return f"Error shortening URL: {e}"
        
def fetch_ticket_price(event_name, ticket_url, delay=2):
    """
    Use Gemini to estimate ticket price from event info.
    Returns the price string or 'TBD' if unknown.
    """
    if pd.isna(ticket_url) or ticket_url.strip() == '':
        return 'N/A'
    
    prompt = f"""Analyze this event and estimate the starting ticket price.

Event: {event_name}
Ticket URL: {ticket_url}

Based on the URL domain and event name:
- If it's likely free (community events, gallery openings, networking), say "Free"
- If you can estimate a price range, give the starting price like "$25" or "$50+"
- If it's a premium/VIP event, estimate accordingly
- If truly unknown, say "TBD"

Respond with ONLY the price (e.g., "Free", "$25", "$100+", "TBD"). Nothing else."""
    
    try:
        time.sleep(delay)  # Rate limiting
        response = model.generate_content(prompt)
        price = response.text.strip()
        return price
    except Exception as e:
        print(f"  Error: {e}")
        return 'TBD'


def standardize_address(address, neighborhood, delay=2):
    """
    Use Gemini to standardize an address format.
    """
    if pd.isna(address) or address.strip() == '':
        return ''
    
    prompt = f"""Standardize this Miami event address to a consistent format.

Original address: {address}
Neighborhood: {neighborhood if pd.notna(neighborhood) else 'Unknown'}

Format as: "Venue Name, Street Address, City, FL ZIP" (if all info available)
- Keep venue name if present
- Add city/state/zip if missing but inferable from neighborhood
- If it's just a venue name with no street address, search to add the address to the venue name or keep as is
- Clean up any formatting issues

Respond with ONLY the standardized address. Nothing else."""
    
    try:
        time.sleep(delay)  # Rate limiting
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        print(f"  Error: {e}")
        return address  # Return original on error


print("✓ Helper functions defined")

✓ Helper functions defined


## Process Ticket Prices

Process events one by one. You can:
- Run a single event to test
- Run a batch of events
- Skip events that already have prices
- Manually override any result

In [34]:
# Test with a single event first
test_idx = 10  # Change this to test different events

row = df.iloc[test_idx]
short_url = shorten_url(row['Tickets Link'])
print(f"Event: {row['Event']}")
print(f"URL: {short_url}")
print(f"Current Price: {row['Ticket Price']}")
print("\nFetching price...")

price = fetch_ticket_price(row['Event'], row['Tickets Link'])
print(f"Estimated Price: {price}")

Event: Arlo Wynwood: Morning Art Run Club
URL: https://tinyurl.com/2az6szwj
Current Price: nan

Fetching price...
Estimated Price: Free


In [53]:
# Process ticket prices for events that don't have one
# Adjust START_IDX and END_IDX to process in batches

START_IDX = 75   # Start from this row
END_IDX = 175    # Process up to (but not including) this row
DELAY = 10       # Seconds between API calls (increase if hitting rate limits)

for idx in range(START_IDX, min(END_IDX, len(df))):
    row = df.iloc[idx]

    short_url = shorten_url(row['Tickets Link'])
    
    # Skip if already has a price
    current_price = row['Ticket Price']
    if pd.notna(current_price) and str(current_price).strip() != '':
        print(f"[{idx}] {row['Event'][:40]} ({short_url})... SKIP (has: {current_price})")
        continue
    
    print(f"[{idx}] {row['Event'][:40]} ({short_url})...", end=' ')
    
    price = fetch_ticket_price(row['Event'], row['Tickets Link'], delay=DELAY)
    df.at[idx, 'Ticket Price'] = price
    
    print(f"-> {price}")

print("\n✓ Batch complete")

[75] IKEA Open House Miami: Private Event (https://tinyurl.com/27zmyz4t)... SKIP (has: TBD)
[76] IKEA Open House Miami: Public Event (Error: 'float' object has no attribute 'startswith')... SKIP (has: N/A)
[77] Immersive Adventure: Basel Edition  (https://tinyurl.com/22r3fvlx)... SKIP (has: $50+)
[78] Ink Miami Art Fair (https://tinyurl.com/2y73ryvt)... SKIP (has: TBD)
[79] Ink Miami Art Fair (https://tinyurl.com/2y73ryvt)... SKIP (has: TBD)
[80] Ink Miami Art Fair (https://tinyurl.com/2y73ryvt)... SKIP (has: TBD)
[81] Ink Miami Art Fair (https://tinyurl.com/2y73ryvt)... SKIP (has: TBD)
[82] John Summit (https://tinyurl.com/2xzqx7np)... SKIP (has: TBD)
[83] Joia Beach (https://tinyurl.com/25yg2u3a)... -> TBD
[84] Little River Studios (https://tinyurl.com/29tq8llh)... -> TBD
[85] LIV (https://tinyurl.com/2bmhd6vv)... -> $100+
[86] Long Feng Art Car: Blond:ish, Desiree, A (Error: HTTPConnectionPool(host='tinyurl.com', port=80): Read timed out. (read timeout=2))... -> $50+
[87] Long Feng 

In [52]:
# View current state of ticket prices
df[['Event', 'Ticket Price', 'Tickets Link']].head(85)

,Event,Ticket Price,Tickets Link
0,"AI + Web3 $44,000 Pitch Competition & VC Netwo...",$50+,https://luma.com/pbs3n297
1,24th Iconic Sagamore Brunch,$175,https://www.eventbrite.com/e/the-24th-iconic-s...
2,33+ Hour Space Basel Closing Party,$50+,https://dice.fm/event/xenqkk-33-hour-space-bas...
3,AfriKin Art Fair,$25,https://afrikin.art/
4,Alo Wellness Sessions,$100+,https://scope-art.com/program
...,...,...,...
80,Ink Miami Art Fair,TBD,https://www.inkartfair.com/
81,Ink Miami Art Fair,TBD,https://www.inkartfair.com/
82,John Summit,TBD,https://posh.vip/e/basel-edition-john-summit-p...
83,Joia Beach,NaN,https://dice.fm/partner/tickets/event/q28d79-s...


In [ ]:
# Manually fix a specific row's ticket price
# ROW_IDX = 5
# df.at[ROW_IDX, 'Ticket Price'] = 'Free'
# print(f"Updated row {ROW_IDX}: {df.iloc[ROW_IDX]['Event']} -> {df.iloc[ROW_IDX]['Ticket Price']}")

## Process Addresses

Standardize address formats one by one or in batches.

In [54]:
# View current addresses
df[['Event', 'Neighborhood', 'Address']].head(20)

,Event,Neighborhood,Address
0,"AI + Web3 $44,000 Pitch Competition & VC Netwo...",NaN,NaN
1,24th Iconic Sagamore Brunch,Miami Beach,Sagamore Hotel South Beach
2,33+ Hour Space Basel Closing Party,Downtown,"34 NE 11th St, Miami, FL 33132"
3,AfriKin Art Fair,North Miami,"Maison AfriKin, Scott Galvin Community Center,..."
4,Alo Wellness Sessions,Miami Beach,"SCOPE Art Show - 801 Ocean Dr, Miami Beach, FL..."
5,Aqua Art Miami,Miami Beach,"Aqua Art Miami at the Aqua Hotel, 1530 Collins..."
6,Aqua Art Miami: VIP Opening,Miami Beach,"Aqua Art Miami at the Aqua Hotel, 1530 Collins..."
7,Arlo Wynwood: AFTER DARK,Wynwood,"2217 Northwest Miami Court, Miami, FL 33127"
8,Arlo Wynwood: Afternoon Art Exhibition,Wynwood,"2217 Northwest Miami Court, Miami, FL 33127"
9,Arlo Wynwood: COFFEE + BEATS,Wynwood,"2217 Northwest Miami Court, Miami, FL 33127"


In [55]:
# Test with a single address first
test_idx = 1  # Change this to test different events

row = df.iloc[test_idx]
print(f"Event: {row['Event']}")
print(f"Neighborhood: {row['Neighborhood']}")
print(f"Current Address: {row['Address']}")
print("\nStandardizing...")

new_address = standardize_address(row['Address'], row['Neighborhood'])
print(f"Standardized: {new_address}")

Event: 24th Iconic Sagamore Brunch
Neighborhood: Miami Beach
Current Address: Sagamore Hotel South Beach

Standardizing...
Standardized: Sagamore Hotel South Beach, 1671 Collins Ave, Miami Beach, FL 33139


In [56]:
# Process addresses in batches
# Adjust START_IDX and END_IDX to process in batches

START_IDX = 0   # Start from this row
END_IDX = 172    # Process up to (but not including) this row
DELAY = 10       # Seconds between API calls

for idx in range(START_IDX, min(END_IDX, len(df))):
    row = df.iloc[idx]
    
    # Skip if no address
    if pd.isna(row['Address']) or row['Address'].strip() == '':
        print(f"[{idx}] {row['Event'][:40]}... SKIP (no address)")
        continue
    
    print(f"[{idx}] {row['Event'][:40]}...")
    print(f"    Before: {row['Address']}")
    
    new_address = standardize_address(row['Address'], row['Neighborhood'], delay=DELAY)
    df.at[idx, 'Address'] = new_address
    
    print(f"    After:  {new_address}")

print("\n✓ Batch complete")

[0] AI + Web3 $44,000 Pitch Competition & VC... SKIP (no address)
[1] 24th Iconic Sagamore Brunch...
    Before: Sagamore Hotel South Beach
    After:  Sagamore Hotel South Beach, 1671 Collins Ave, Miami Beach, FL 33139
[2] 33+ Hour Space Basel Closing Party...
    Before: 34 NE 11th St, Miami, FL 33132
    After:  34 NE 11th St, Miami, FL 33132
[3] AfriKin Art Fair...
    Before: Maison AfriKin, Scott Galvin Community Center, 1600 NE 126th Street, North Miami, FL 33181
    After:  Maison AfriKin, Scott Galvin Community Center, 1600 NE 126th Street, North Miami, FL 33181
[4] Alo Wellness Sessions...
    Before: SCOPE Art Show - 801 Ocean Dr, Miami Beach, FL 33139
    After:  SCOPE Art Show, 801 Ocean Dr, Miami Beach, FL 33139
[5] Aqua Art Miami...
    Before: Aqua Art Miami at the Aqua Hotel, 1530 Collins Avenue, Miami Beach, FL 33139
    After:  Aqua Art Miami at the Aqua Hotel, 1530 Collins Avenue, Miami Beach, FL 33139
[6] Aqua Art Miami: VIP Opening...
    Before: Aqua Art Miami at

In [ ]:
# Manually fix a specific row's address
# ROW_IDX = 5
# df.at[ROW_IDX, 'Address'] = '123 Main St, Miami, FL 33101'
# print(f"Updated row {ROW_IDX}")

## Review & Save

In [57]:
# Review all changes
print("Final data preview:")
df[['Event', 'Ticket Price', 'Address']].head(30)

Final data preview:


,Event,Ticket Price,Address
0,"AI + Web3 $44,000 Pitch Competition & VC Netwo...",$50+,NaN
1,24th Iconic Sagamore Brunch,$175,"Sagamore Hotel South Beach, 1671 Collins Ave, ..."
2,33+ Hour Space Basel Closing Party,$50+,"34 NE 11th St, Miami, FL 33132"
3,AfriKin Art Fair,$25,"Maison AfriKin, Scott Galvin Community Center,..."
4,Alo Wellness Sessions,$100+,"SCOPE Art Show, 801 Ocean Dr, Miami Beach, FL ..."
5,Aqua Art Miami,$25,"Aqua Art Miami at the Aqua Hotel, 1530 Collins..."
6,Aqua Art Miami: VIP Opening,$100+,"Aqua Art Miami at the Aqua Hotel, 1530 Collins..."
7,Arlo Wynwood: AFTER DARK,$50+,"2217 NW Miami Court, Miami, FL 33127"
8,Arlo Wynwood: Afternoon Art Exhibition,Free,"2217 Northwest Miami Court, Miami, FL 33127"
9,Arlo Wynwood: COFFEE + BEATS,Free,"2217 NW Miami Court, Miami, FL 33127"


In [63]:
# Check for any remaining empty ticket prices
empty_prices = df[df['Ticket Price'].isna() | (df['Ticket Price'] == '')]
print(f"Events still missing ticket prices: {len(empty_prices)}")
if len(empty_prices) > 0:
    print(empty_prices[['Event', 'Tickets Link']])

Events still missing ticket prices: 0


In [61]:
# Save the preprocessed CSV
output_path = '../art-basel-2025.csv'
df.to_csv(output_path, index=False)
print(f"✓ Saved to {output_path}")
print(f"\nNext step: Run 'npm run update-data' to convert to JSON")

✓ Saved to art-basel-2025-preprocessed.csv

Next step: Run 'npm run update-data' to convert to JSON


In [65]:
# Optional: Overwrite the original CSV instead
# df.to_csv('../art-basel-2025.csv', index=False)
# print("✓ Saved to art-basel-2025.csv (original overwritten)")

✓ Saved to art-basel-2025.csv (original overwritten)
